# CS 3110/5110: Data Privacy
## Homework 5

In [3]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(qs, sensitivity, epsilon):
    return [laplace_mech(q, sensitivity, epsilon) for q in qs]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(qs, sensitivity, epsilon, delta):
    return [gaussian_mech(q, sensitivity, epsilon, delta) for q in qs]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1 (5 points)

Implement the `dp_occupation_histogram` function below. It should return a differentially private histogram over the `Occupation` column in the `adult` dataset. Your function should have a total privacy cost of `epsilon` and should use parallel composition.

In [11]:
def dp_occupation_histogram(epsilon):
    hist = adult.groupby(["Occupation"]).size()
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))

dp_occupation_histogram(1.0)

Occupation
Adm-clerical         3770.741924
Armed-Forces            6.741340
Craft-repair         4097.878963
Exec-managerial      4066.596152
Farming-fishing       992.844939
Handlers-cleaners    1370.434920
Machine-op-inspct    2002.095738
Other-service        3293.988077
Priv-house-serv       148.302365
Prof-specialty       4139.775410
Protective-serv       649.303542
Sales                3649.131933
Tech-support          926.903604
Transport-moving     1596.420362
dtype: float64

In [10]:
# TEST CASE
hist = adult['Occupation'].value_counts()
dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 2
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [np.abs(dp_occupation_histogram(0.01)-hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/0.01) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 150
assert stats.wasserstein_distance(all_dp_results, spec) > 50

C:\Users\gresh\AppData\Local\Temp\ipykernel_15100\2649794634.py:3: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
C:\Users\gresh\AppData\Local\Temp\ipykernel_15100\2649794634.py:3: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
C:\Users\gresh\AppData\Local\Temp\ipykernel_15100\2649794634.py:3: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
C:\Users\gresh\AppData\Local\Temp\ipykernel_15100\2649794634.py:3: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  dp_results = [np.abs(dp_occupation_histogram(1.0)-hist) for _ in range(200)]
C:\Users\gre

AssertionError: 

## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_occupation_histogram` satisfies `epsilon`-differential privacy.
By the rules of parallel composition, dp_occupation_histogram satisfies epsilon. The adult dataset is split into disjoint subsets and the Laplace mech is applied with epsilon to each subset.
- How would your results change (in terms of accuracy) if you used sequential composition instead of parallel composition to implement `dp_occupation_histogram`, and why?
- If you partition the data by *both* occupation and age (i.e. a contingency table), would parallel composition still apply? Why or why not?
- If you partition the data by both occupation and age, what would happen to accuracy, and why?
Decrease, sparser data

YOUR ANSWER HERE

## Question 3 (5 points)

Implement the `dp_3d_histogram` function below. It should return a differentially private 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns (in that order) in the `adult` dataset (i.e. one count for each combination of occupation, education, and sex). As in Question 1, your function should have a total privacy cost of `epsilon` and should use parallel composition.

*Hint*: `value_counts` works on multiple columns.

In [12]:
def dp_3d_histogram(epsilon):
    hist = adult.groupby(["Occupation", "Education", "Sex"]).size()
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))

dp_3d_histogram(1.0)

Occupation        Education     Sex   
Adm-clerical      10th          Female     21.791872
                                Male       17.261511
                  11th          Female     45.462773
                                Male       17.998921
                  12th          Female     24.918109
                                             ...    
Transport-moving  HS-grad       Male      773.305968
                  Masters       Male        9.564923
                  Prof-school   Male        2.428896
                  Some-college  Female     20.880642
                                Male      263.111028
Length: 361, dtype: float64

In [13]:
# TEST CASE
hist = adult[['Occupation', 'Education', 'Sex']].value_counts()
dp_results = [(dp_3d_histogram(1.0)-hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 0.3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [(dp_3d_histogram(0.1)-hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1/0.1) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_3d_histogram` satisfies `epsilon`-differential privacy.
- Which histogram do you expect to have higher relative error (percent error): the 1-dimensional one from Question 1, or the 3-dimensional one from Question 3? Why?
The 3-dimensional one has higher error.

YOUR ANSWER HERE

## Question 5 (5 points)

Implement the `dp_three_histograms` function below. It should return a tuple of three differentially private histograms:

1. A 1-dimensional histogram over the `Occupation` column
2. A 2-dimensional histogram over the `Occupation` and `Education` columns
3. A 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns

Your function should have a total privacy cost of `epsilon`.

Hint: take care with composition!

In [ ]:
def dp_three_histograms(epsilon):
    hists = []
    hist_cols = (["Occupation"], ["Occupation", "Education"], ["Occupation", "Education", "Sex"])

    for hc in hist_cols:
        hist = adult.groupby(hc).size()
        hists.append(hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon)))

    return hists

dp_three_histograms(1.0)

In [ ]:
# TEST CASE
hists = [adult['Occupation'].value_counts(),
         adult[['Occupation', 'Education']].value_counts(),
         adult[['Occupation', 'Education', 'Sex']].value_counts()]
dp_results = [dp_three_histograms(1.0) for _ in range(100)]
assert all(len(r) == 3 for r in dp_results)

for i, hist in enumerate(hists):
    all_dp_results = [item for r in dp_results for item in (r[i]-hist).dropna()]
    spec = [np.random.laplace(loc=0, scale=3/1.0) for _ in range(2000)]
    assert stats.wasserstein_distance(all_dp_results, spec) < 1
    assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 6 (5 points)

Using the 3-point structure described in lecture, argue that your solution for `dp_three_histograms` satisfies `epsilon`-differential privacy. Be precise about composition.

YOUR ANSWER HERE